---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [4]:
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def pobierz_fakt(i):
    response = requests.get(CAT_API_URL)
    return response.json().get("fact")

def pobieranie_sekwencyjne():
    start = time.time()

    for i in range(20):
        fakt = requests.get(CAT_API_URL).json().get("fact")

    czas = time.time() - start
    print(f"Czas sekwencyjny: {czas:.2f}s")
    return czas

def pobieranie_wielowatkowe():
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor() as executor:
        fakty = list(executor.map(pobierz_fakt, range(20)))

    czas = time.time() - start
    print(f"Czas wielowątkowy: {czas:.2f}s")
    return czas

czas1 = pobieranie_sekwencyjne()
czas2 = pobieranie_wielowatkowe()

print(f"Sekwencyjnie: {czas1:.2f}s")
print(f"Wielowątkowo: {czas2:.2f}s")

Czas sekwencyjny: 4.13s
Czas wielowątkowy: 0.40s
Sekwencyjnie: 4.13s
Wielowątkowo: 0.40s


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [5]:
import queue
import threading
import time

LICZBA_ELEMENTOW = 20

kolejka = queue.Queue()
parzyste = []
nieparzyste = []

przetworzone = 0
licznik_lock = threading.Lock()


def producent():
    for liczba in range(1, LICZBA_ELEMENTOW + 1):
        kolejka.put(liczba)
        print(f"Producent dodał do kolejki: {liczba}")
        time.sleep(0.01)


def konsument(nazwa, czy_parzysta, wyniki):
    global przetworzone

    while True:
        with licznik_lock:
            if przetworzone >= LICZBA_ELEMENTOW:
                break

        try:
            liczba = kolejka.get(timeout=0.1)
        except queue.Empty:
            continue

        if (liczba % 2 == 0) == czy_parzysta:
            wyniki.append(liczba)
            print(f"{nazwa} pobrał: {liczba}")

            with licznik_lock:
                przetworzone += 1
        else:
            kolejka.put(liczba)
            time.sleep(0.01)

        kolejka.task_done()


watek_producent = threading.Thread(target=producent)
watek_parzysty = threading.Thread(
    target=konsument,
    args=("Konsument parzysty", True, parzyste)
)
watek_nieparzysty = threading.Thread(
    target=konsument,
    args=("Konsument nieparzysty", False, nieparzyste)
)

watek_producent.start()
watek_parzysty.start()
watek_nieparzysty.start()

watek_producent.join()
watek_parzysty.join()
watek_nieparzysty.join()

print("\nLiczby parzyste:", sorted(parzyste))
print("Liczby nieparzyste:", sorted(nieparzyste))
print("Liczba przetworzonych elementów:", len(parzyste) + len(nieparzyste))


Producent dodał do kolejki: 1
Konsument nieparzysty pobrał: 1
Producent dodał do kolejki: 2
Konsument parzysty pobrał: 2
Producent dodał do kolejki: 3
Konsument nieparzysty pobrał: 3
Producent dodał do kolejki: 4
Konsument parzysty pobrał: 4
Producent dodał do kolejki: 5
Konsument nieparzysty pobrał: 5
Producent dodał do kolejki: 6
Konsument parzysty pobrał: 6
Producent dodał do kolejki: 7
Konsument nieparzysty pobrał: 7
Producent dodał do kolejki: 8
Konsument parzysty pobrał: 8
Producent dodał do kolejki: 9
Konsument nieparzysty pobrał: 9
Producent dodał do kolejki: 10
Konsument parzysty pobrał: 10
Producent dodał do kolejki: 11
Konsument nieparzysty pobrał: 11
Producent dodał do kolejki: 12
Konsument parzysty pobrał: 12
Producent dodał do kolejki: 13
Konsument nieparzysty pobrał: 13
Producent dodał do kolejki: 14
Konsument parzysty pobrał: 14
Producent dodał do kolejki: 15
Konsument nieparzysty pobrał: 15
Producent dodał do kolejki: 16
Konsument parzysty pobrał: 16
Producent dodał do

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [2]:
import multiprocessing
import time
from lab2_functions import calculate_power_sum

POCZATEK = 1
KONIEC = 10_000
LICZBA_PROCESOW = 4


def oblicz_sekwencyjnie(liczby):
    start = time.time()
    wyniki = [calculate_power_sum(liczba) for liczba in liczby]
    czas = time.time() - start
    return wyniki, czas


def oblicz_rownolegle(liczby):
    start = time.time()

    with multiprocessing.Pool(processes=LICZBA_PROCESOW) as pool:
        wyniki = pool.map(calculate_power_sum, liczby)

    czas = time.time() - start
    return wyniki, czas


if __name__ == "__main__":
    liczby = list(range(POCZATEK, KONIEC + 1))

    wyniki_sekwencyjne, czas_sekwencyjny = oblicz_sekwencyjnie(liczby)
    wyniki_rownolegle, czas_rownolegly = oblicz_rownolegle(liczby)

    print(f"Liczba obliczeń: {len(wyniki_rownolegle)}")
    print(f"Czas sekwencyjny: {czas_sekwencyjny:.2f}s")
    print(f"Czas równoległy: {czas_rownolegly:.2f}s")

    if czas_rownolegly > 0:
        print(f"Przyspieszenie: {czas_sekwencyjny / czas_rownolegly:.2f}x")

    print("Zgodność wyników:", wyniki_sekwencyjne == wyniki_rownolegle)
    print("Pierwszy wynik:", wyniki_rownolegle[0])
    print("Ostatni wynik:", wyniki_rownolegle[-1])

Liczba obliczeń: 10000
Czas sekwencyjny: 0.31s
Czas równoległy: 0.17s
Przyspieszenie: 1.83x
Zgodność wyników: True
Pierwszy wynik: 100
Ostatni wynik: 10001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010000
